# Lesson 40 · Spectrum analysis in practice — printing out a sound's ingredient list: magnitude and phase spectra, and frequency resolution

**Goal**: start by using the FFT as a "printer for a sound's ingredient list": learn to read the magnitude and phase spectra, nail down $\Delta f = sr/N$, then understand why a real signal only needs half of its spectrum.

> If the phase is hard to read, think back to L06: `np.angle` wraps around; for a real signal you usually only interpret the positive-frequency half.

🔗 Aurora connection: `aurora.audio.stft.magnitude_spectrogram()` does the same frequency-axis mapping internally — this section takes that process apart so you can build it yourself.

← **Previous lesson** · [L39 · Writing the FFT from scratch](L39_fft_implement.ipynb)

> Last lesson covered **writing the FFT from scratch**: a recursive Cooley–Tukey implementation, error < 1e-10 against numpy.fft.
> This lesson explores **spectrum analysis in practice**.

## This lesson's story: what is a spectrum like? Print out the sound's "ingredient list"

Sip a glass of fruit juice and you can taste whether there's orange, apple or lemon in it. Your ears do something similar with sound — the brain breaks a mixed sound wave into "ingredients" at different frequencies and tells you whether it's a low bass guitar or a high flute.

That's exactly what the FFT does: it turns a time-domain signal into a **spectrum** — the strength of every frequency at a glance.

But there's a detail that's easy to miss: in the FFT output `X[k]`, `k` is only "which frequency bin", not Hz. To turn a bin number into Hz, you need:

$$\Delta f = \frac{f_s}{N}$$

- `fₛ`: the sample rate (Hz)
- `N`: the number of FFT points
- `Δf`: the spacing between neighboring frequency bins (Hz/bin), also called the **frequency resolution**

**Worked example**: `fₛ=8000, N=256` → `Δf=31.25 Hz/bin`
The 440 Hz piano note A4 lands in bin `k = round(440/31.25) = 14`, whose exact frequency is `14×31.25 = 437.5 Hz`.

440÷31.25 comes out as **14.08**, not a whole number — that's not a mistake; it's a reminder that 440 Hz doesn't land "exactly" on the center of any bin, so the FFT can only "file" it under the nearest bin. What rounding 14.08 to 14 means, and how wide a stretch of frequencies that bin really "stands for", is spelled out with concrete numbers in section 1 of this lesson.

Your task in this section: implement `frequency_bins(N, sr)` — return the one-sided frequency axis, shape `(N//2+1,)`, from 0 Hz up to Nyquist.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine
from aurora.audio.transforms import (
    fft as aurora_fft,
    next_power_of_two,
    fft_frequencies as aurora_fft_frequencies,
)
# aurora_fft is Aurora's Cooley-Tukey FFT (the iterative twin of the one you wrote in L39); it needs a power-of-2 input length,
# so a signal of any other length must first be zero-padded with next_power_of_two before it goes in.


## 1. Bin spacing and the frequency axis

The FFT divides `[0, sr)` Hz evenly into N parts, so the frequency spacing between neighboring bins is:

```
Δf = sr / N   (Hz/bin)
```

The frequency of bin k is:

```
freqs[k] = k * sr / N
```

In other words, the full frequency axis is `freqs = np.arange(N) * sr / N`, running from 0 Hz all the way up to `(N-1)*sr/N ≈ sr`.
The bigger N is, the smaller Δf, and the finer the frequency resolution.

### Why is Δf = sr/N? The maths behind it

Imagine a recording T seconds long (from t=0 to t=T), at sample rate sr (samples per second), with N samples collected in all.
So: **T = N / sr** (seconds)

From Fourier analysis, when we take the Fourier transform of a signal of length T, the frequency-domain resolution (the spacing between neighboring frequency components) is:
$$\Delta f = \frac{1}{T} = \frac{1}{N/sr} = \frac{sr}{N}$$

**What it means physically**:
- The longer the recording (at a fixed sample rate, the bigger N), the smaller the frequency spacing Δf you can resolve → the finer the frequency resolution
- The higher the sample rate (the bigger sr), the wider the range of frequencies covered (up to sr/2)

**The range of the full frequency axis**: the FFT covers the frequency range [0, sr). The physical frequency of bin k is just **k times the basic spacing**:
$$f_k = k \cdot \Delta f = k \cdot \frac{sr}{N}$$

### What does a fractional bin mean? — each bin covers a range, not a point

Back to the question from the story: 440÷31.25 = **14.08**, rounded to 14. Where did that 0.08 go?

An analogy first: think of the frequency axis as a ruler with **no tick marks, only cells**. Each cell (bin) is Δf wide, and standing at its center is a "representative frequency", k·Δf. Any sound that falls into the cell, the FFT tells: "you're this representative frequency" — even if it's actually a little off.

So how wide is a cell, and where are its edges? Picture two neighboring representative frequencies: bin 14's center is at `14×Δf`, bin 15's at `15×Δf`, exactly one Δf apart. The **midpoint** between these two centers is the boundary between the two cells — and its distance from bin 14's center is exactly half of Δf:

$$\text{boundary} = \frac{14\Delta f + 15\Delta f}{2} = 14\Delta f + \frac{\Delta f}{2}$$

Likewise, the boundary between bin 14 and bin 13 on its left is at `14Δf − Δf/2`. So the range of frequencies that bin k really "covers" is:

$$\left[\, k\cdot\Delta f - \frac{\Delta f}{2},\ \ k\cdot\Delta f + \frac{\Delta f}{2} \,\right)$$

That is, **center ± half a Δf**, exactly one Δf wide (the cells sit end to end, with no overlaps and no gaps).

**Let's run Samuel's example through it**: `sr=8000, N=256` → `Δf = 31.25 Hz`.

- bin 14's center = `14 × 31.25 = 437.5 Hz`
- half a Δf = `31.25 / 2 = 15.625 Hz`
- the range bin 14 covers = `[437.5 − 15.625, 437.5 + 15.625) = [421.875, 453.125) Hz`

440 Hz falls inside `[421.875, 453.125)`, which is why it gets "rounded" into bin 14 — the frequency the FFT reports is 437.5 Hz, not 440 Hz itself. That's what the 0.08 in 14.08 really means: **the distance from 440 Hz to bin 14's center is 0.08×2 ≈ 16% of half a cell's width** — still inside the cell, so it's filed under the nearest bin, 14.

**This also explains why a longer recording (bigger N) pins frequencies down better**: bigger N → smaller Δf → narrower cells → frequencies as close together as 440 Hz and 441 Hz have a better chance of landing in different cells and being told apart. (Zero-padding also narrows the cells, but it adds no real resolution — see the zero-padding discussion further down.) Below, we check these numbers in code.

In [ ]:
# Demo: what a fractional bin really means — how wide a frequency range does bin 14 cover?
sr_demo, N_demo2 = 8000, 256
delta_f_demo = sr_demo / N_demo2
target_freq = 440

k_bin = round(target_freq / delta_f_demo)
bin_center = k_bin * delta_f_demo
half_width = delta_f_demo / 2
lo, hi = bin_center - half_width, bin_center + half_width

print(f"Δf = sr/N = {sr_demo}/{N_demo2} = {delta_f_demo} Hz/bin")
print(f"{target_freq} Hz → k = round({target_freq}/{delta_f_demo}) = {k_bin}")
print(f"center frequency of bin {k_bin} = {k_bin} × {delta_f_demo} = {bin_center} Hz")
print(f"frequency range bin {k_bin} actually covers = [center − Δf/2, center + Δf/2) = [{lo}, {hi}) Hz")
print(f"→ {target_freq} Hz falls inside [{lo}, {hi}): {lo <= target_freq < hi}")
print(f"→ any sound between {lo} and {hi} Hz is filed by the FFT under bin {k_bin} and reported as {bin_center} Hz")

assert lo <= target_freq < hi
assert np.isclose(hi - lo, delta_f_demo)

In [ ]:
sr = 8000
for N in [8, 16, 64]:
    delta_f = sr / N
    print(f'N={N:4d}  Δf = {delta_f:.1f} Hz')

## 2. Real-signal symmetry and the Nyquist frequency

When the input `x[n]` is a real sequence, the FFT has conjugate symmetry:

```
|X[k]| = |X[N-k]|
```

This means the second half (k > N/2) is completely determined by the first half and carries no extra information.
The useful frequency range is **0 to sr/2**, which corresponds to bin indices 0 to N//2 (N//2+1 bins in all).
The highest frequency, `sr/2`, is called the **Nyquist frequency** — the sampling theorem (Nyquist–Shannon sampling theorem) requires the signal to contain no components above it, otherwise aliasing occurs.

### Why does a real input give conjugate symmetry?

It's no coincidence — it's a deep property of the Fourier transform. Let's look at it with a small example.

**The DFT definition**:
$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j2\pi kn/N}$$

When x[n] is real, let's look at X[N-k] (that is, replace k with N-k):
$$X[N-k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j2\pi(N-k)n/N}$$

Notice that $e^{-j2\pi(N-k)n/N} = e^{-j2\pi Nn/N} \cdot e^{j2\pi kn/N} = 1 \cdot e^{j2\pi kn/N} = e^{j2\pi kn/N}$ (because $e^{-j2\pi n}=1$)

So:
$$X[N-k] = \sum_{n=0}^{N-1} x[n] \cdot e^{j2\pi kn/N} = \text{conj}\left(\sum_{n=0}^{N-1} x[n] \cdot e^{-j2\pi kn/N}\right) = \text{conj}(X[k])$$

The last step uses the fact that when x[n] is real, the conjugate of $\sum x[n] \cdot e^{j\theta_n}$ equals $\sum x[n] \cdot e^{-j\theta_n}$.

**Conclusion**: real input $\Rightarrow$ $X[N-k] = X[k]^*$ (conjugate symmetry) $\Rightarrow$ magnitude $|X[N-k]| = |X[k]|$, phase $\angle X[N-k] = -\angle X[k]$

In [ ]:
# Check the symmetry (using aurora_fft, the L39-style FFT; N=16 = 2^4, so no padding needed)
sr = 8000
N = 16
rng = np.random.default_rng(42)
x = rng.standard_normal(N)
X = aurora_fft(x)
# confirm aurora_fft agrees with numpy
assert np.allclose(aurora_fft(x), np.fft.fft(x)), "aurora_fft should match np.fft.fft exactly"
print('|X[1]| vs |X[N-1]|:', np.abs(X[1]).round(6), np.abs(X[N-1]).round(6))
print('|X[2]| vs |X[N-2]|:', np.abs(X[2]).round(6), np.abs(X[N-2]).round(6))
print('Nyquist frequency =', sr // 2, 'Hz, at bin', N // 2)


In [ ]:
# Worked example: N=4, the real signal x = [1, 0, -1, 0], to see conjugate symmetry concretely
x_small = np.array([1.0, 0.0, -1.0, 0.0])
X_small = np.fft.fft(x_small)  # using numpy's standard FFT

print("="*60)
print("A concrete example of conjugate symmetry: N=4, x=[1, 0, -1, 0]")
print("="*60)
for k_val in range(4):
    print(f"X[{k_val}] = {X_small[k_val]:.4f}")

print("\nCheck conjugate symmetry X[N-k] = conj(X[k]):")
for k_val in range(1, 4):
    X_Nk = X_small[4 - k_val]
    conj_Xk = np.conj(X_small[k_val])
    match = "✓" if np.isclose(X_Nk, conj_Xk) else "✗"
    print(f"  X[{4-k_val}] = {X_Nk:.4f}  vs  conj(X[{k_val}]) = {conj_Xk:.4f}  {match}")

print("\nMagnitude symmetry |X[N-k]| = |X[k]|:")
for k_val in range(1, 4):
    mag_Nk = np.abs(X_small[4 - k_val])
    mag_k = np.abs(X_small[k_val])
    match = "✓" if np.isclose(mag_Nk, mag_k) else "✗"
    print(f"  |X[{4-k_val}]| = {mag_Nk:.4f}  vs  |X[{k_val}]| = {mag_k:.4f}  {match}")

print("\nPhase antisymmetry angle(X[N-k]) = -angle(X[k]):")
for k_val in range(1, 4):
    phase_Nk = np.angle(X_small[4 - k_val])
    phase_k = np.angle(X_small[k_val])
    match = "✓" if np.isclose(phase_Nk, -phase_k, atol=1e-10) else "✗"
    print(f"  angle(X[{4-k_val}]) = {phase_Nk:.4f}  vs  -angle(X[{k_val}]) = {-phase_k:.4f}  {match}")

## 3. The magnitude spectrum, the power spectrum and the dB spectrum

Applying different scalar transforms to the FFT coefficients `X[k]` gives three commonly used spectra:

```
magnitude spectrum  A[k] = |X[k]|
power spectrum      P[k] = |X[k]|²
dB spectrum         D[k] = 20 * log10(|X[k]| + eps)
```

### Why is the dB coefficient 20 and not 10?

It comes from the relationship between power and amplitude. Power P relates to amplitude A by:
$$P = A^2$$

If we take the log of the power (in dB):
$$P_{\text{dB}} = 10 \log_{10}(P) = 10 \log_{10}(A^2) = 10 \cdot 2 \log_{10}(A) = 20 \log_{10}(A)$$

So **20 log₁₀(A) is really "the power spectrum expressed in dB"**.

**Why use dB at all?**
- Human hearing perceives sound intensity roughly logarithmically (twice as loud ≈ 10 dB more)
- Audio has a huge dynamic range (from barely-there background noise to an ear-splitting scream), so on a linear scale it's hard to see the peaks and the valleys at the same time
- dB **compresses** that huge range into something you can see, bringing the details out

**The logarithm "turns multiplication into addition"**:
If two spectral components need to be multiplied (say, the effect of filters in series), in the dB domain they simply add:
$$A_{\text{dB}} = 20\log_{10}(A_1 \cdot A_2) = 20\log_{10}(A_1) + 20\log_{10}(A_2)$$

That makes it easy to stack up the effects of several processing steps intuitively when designing a system.

`eps` (usually `1e-8`) keeps you from taking the log of zero.

In [ ]:
sr = 8000
duration = 0.5
x_demo = sine(440, duration, sr)  # a pure 440 Hz tone
N_demo = len(x_demo)          # 4000, not a power of 2
N_fft  = next_power_of_two(N_demo)  # 4096 = 2^12, the length after zero-padding
x_padded = np.zeros(N_fft)
x_padded[:N_demo] = x_demo
X_demo = aurora_fft(x_padded)  # the L39-style FFT (aurora.audio.transforms.fft)
assert np.allclose(X_demo, np.fft.fft(x_padded)), "aurora_fft should match np.fft.fft"
half = N_fft // 2 + 1
freqs_demo = np.arange(half) * sr / N_fft

A = np.abs(X_demo[:half])
P = A ** 2
D = 20 * np.log10(A + 1e-8)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].plot(freqs_demo, A); axes[0].set_title('Magnitude spectrum'); axes[0].set_xlabel('Hz')
axes[1].plot(freqs_demo, P); axes[1].set_title('Power spectrum'); axes[1].set_xlabel('Hz')
axes[2].plot(freqs_demo, D); axes[2].set_title('dB spectrum'); axes[2].set_xlabel('Hz')
plt.tight_layout()
plt.show()


## 3b. The phase spectrum

A complex FFT coefficient `X[k]` carries two pieces of information:

```
X[k] = |X[k]| · e^{jφ[k]}
magnitude spectrum  A[k] = |X[k]|        → how strong each frequency component is
phase spectrum      φ[k] = angle(X[k])   → each frequency component's initial phase (radians)
```

### Understanding Euler's formula and the complex exponential e^{jφ}

Why use `e^{jφ}` to represent phase? It comes from Euler's formula:

$$e^{j\theta} = \cos(\theta) + j\sin(\theta)$$

where j is the imaginary unit (j² = -1).

**An intuitive example**:
- $e^{j \cdot 0} = \cos(0) + j\sin(0) = 1$ (the positive real axis on the complex plane)
- $e^{j \cdot \pi/2} = \cos(\pi/2) + j\sin(\pi/2) = j$ (the positive imaginary axis)
- $e^{j \cdot \pi} = \cos(\pi) + j\sin(\pi) = -1$ (the negative real axis)
- $e^{j \cdot 2\pi} = \cos(2\pi) + j\sin(2\pi) = 1$ (once around the circle, back to the start)

So `e^{jφ}` really just rotates by an angle φ on the complex plane. **Multiplying an amplitude A by `e^{jφ}` means: a complex number of size A pointing in direction φ.**

**Phase antisymmetry of real signals**:

$$\varphi[k] = -\varphi[N-k]   \text{(the counterpart of the magnitude symmetry)}$$

This means the second half of the phase spectrum is a negated mirror image of the first half — fully consistent with conjugate symmetry. If $X[k] = A[k] e^{j\varphi[k]}$, then $X[N-k] = X[k]^* = A[k] e^{-j\varphi[k]}$, so $\varphi[N-k] = -\varphi[k]$.

**What it means physically**: for a single-frequency tone $\cos(2\pi f_0 t + \theta_0)$ that lands exactly on a bin, the FFT phase at f₀'s bin
is the initial phase $\theta_0$ (for a sine, $\sin(2\pi f_0 t + \theta_0)$, it is $\theta_0 - \pi/2$, since a sine is a cosine a quarter turn behind). The phase spectrum plays a central role in audio watermarking, the phase vocoder and
time-stretching.

In [ ]:
# Phase-spectrum example: a 440 Hz + 880 Hz mix (zero-padded to a power of 2)
sr = 8000
x_phase = sine(440, 0.5, sr) + sine(880, 0.5, sr)
N_p = len(x_phase)
N_fft_p = next_power_of_two(N_p)
x_phase_padded = np.zeros(N_fft_p)
x_phase_padded[:N_p] = x_phase
X_phase = aurora_fft(x_phase_padded)
half_p = N_fft_p // 2 + 1
freqs_p = np.arange(half_p) * sr / N_fft_p

phi = np.angle(X_phase[:half_p])   # φ[k] = arctan2(Im, Re), in radians

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))
ax1.plot(freqs_p, np.abs(X_phase[:half_p]))
ax1.set_xlim(0, 1200)
ax1.set_title('Magnitude spectrum (for reference)')
ax1.set_ylabel('Magnitude')
ax2.plot(freqs_p, phi)
ax2.set_xlim(0, 1200)
ax2.axhline(0, color=plt.rcParams["axes.edgecolor"], linewidth=0.5)
ax2.set_title('Phase spectrum φ[k] = angle(X[k])')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Phase (rad)')
plt.tight_layout()
plt.show()

# Check the phase antisymmetry of a real signal: φ[k] ≈ −φ[N_fft_p − k]
X_full = aurora_fft(x_phase_padded)
phi_full = np.angle(X_full)
k_test = 5
print(f'φ[{k_test}] = {phi_full[k_test]:.4f}  rad')
print(f'−φ[N−{k_test}] = {-phi_full[N_fft_p - k_test]:.4f}  rad')
print('antisymmetry holds:', np.isclose(phi_full[k_test], -phi_full[N_fft_p - k_test], atol=1e-10))


In [ ]:
# Phase of a single-frequency sine: sin(2π·440t + π/6), initial phase π/6 (30°)
sr = 8000
freq = 440
init_phase = np.pi / 6  # initial phase = 30° = π/6 radians
duration = 0.5

# generate the signal: sin(2π·f·t + φ₀)
t = np.arange(int(sr * duration)) / sr
x_phase_example = np.sin(2 * np.pi * freq * t + init_phase)

# FFT (zero-padded to a power of 2)
N = len(x_phase_example)
N_fft = next_power_of_two(N)
x_padded = np.zeros(N_fft)
x_padded[:N] = x_phase_example
X = np.fft.fft(x_padded)

# find the bin for 440 Hz
bin_440 = round(freq / (sr / N_fft))
fft_phase = np.angle(X[bin_440])

print("="*60)
print("Phase of a single-frequency sine")
print("="*60)
print(f"signal: x(t) = sin(2π·{freq}·t + {init_phase:.4f})")
print(f"initial phase: {init_phase:.4f} rad = {np.degrees(init_phase):.1f}°")
print(f"\nFFT result:")
print(f"  bin for {freq} Hz: {bin_440}")
print(f"  phase of the FFT coefficient (from angle()): {fft_phase:.4f} rad = {np.degrees(fft_phase):.1f}°")
print(f"\nComparison:")
print(f"  initial phase {init_phase:.4f} rad")
print(f"  FFT phase {fft_phase:.4f} rad")
print(f"  difference: {abs(fft_phase - init_phase):.6f} rad (not zero: for a sine on an exact bin, expect θ₀ − π/2; here 440 Hz sits between bins and the zero-padding shifts it further)")
print("\n💡 Note: angle() measures phase against a cosine, so a sine starts a quarter turn (π/2) behind; off-bin tones and zero-padding shift the reading as well.")
print("   Still, the phase information is all there in X[k], and that is what the phase vocoder and time-stretching rely on.")

## 4. ✏️ Implement `frequency_bins(N, sr)`

**What it means physically**: the frequency of the k-th FFT coefficient = `k × sr / N` (Hz).

**Why return only half?**
The FFT of a real signal has conjugate symmetry: `X[N-k] = X[k]*`. A one-sided spectrum only needs `k = 0..N//2`, `N//2+1` points in all, from 0 Hz to Nyquist (sr/2).

**Two-step implementation route**:

| Step | Code | Notes |
|---|---|---|
| 1 | `freqs_full = np.arange(N) * sr / N` | the full frequency axis, length N |
| 2 | `return freqs_full[:N//2 + 1]` | keep the first half (including Nyquist) |

**Acceptance criteria**:
- `frequency_bins(8, 8000)[0] == 0.0` (the DC component)
- `frequency_bins(8, 8000)[-1] == 4000.0` (Nyquist)
- `frequency_bins(8, 8000)` has length `8//2+1 = 5`

In [ ]:
def frequency_bins(N: int, sr: int) -> np.ndarray:
    """Return the frequency axis of a real signal's FFT, shape (N//2+1,), in Hz."""
    # ✏️ TODO: build the full frequency axis, then keep the first N//2+1 values
    raise NotImplementedError("TODO: freqs_full = np.arange(N) * sr / N; return freqs_full[:N // 2 + 1]")

In [ ]:
# Numeric check
try:
    result = frequency_bins(8, 8000)
    if result is ... or result is None:
        print('⬜ Implement frequency_bins first, then run this cell')
    else:
        assert result[-1] == 4000, f'Nyquist should be 4000, got {result[-1]}'
        assert len(result) == 5, f'length should be 5, got {len(result)}'
        assert result[0] == 0.0, 'the first bin should be 0 Hz'
        assert np.allclose(result, [0., 1000., 2000., 3000., 4000.])
        # 🔗 cross-check: should match aurora.audio.transforms.fft_frequencies numerically
        ref = aurora_fft_frequencies(8000, 8)
        assert np.allclose(result, ref), (
            f'disagrees with aurora_fft_frequencies: got {result}, expected {ref}'
        )
        print('✅ frequency_bins passes every check (including the cross-check against aurora_fft_frequencies)')
except NotImplementedError:
    print('⬜ frequency_bins isn\'t implemented yet; finish the TODO, then run this cell')
except TypeError:
    print('⬜ frequency_bins returned something that isn\'t an array; finish the TODO')

## 5. Parameter experiment: locating two peaks

Generate a mix of **440 Hz** and **880 Hz** and plot its one-sided magnitude spectrum.

**What to expect**:
- the two spikes show up right around the bins for 440 Hz and 880 Hz
- the longer the `duration` (the bigger N), the narrower the peaks and the more precisely they're located (smaller Δf)
- change `duration` to `0.01` (very small N) and the peaks get visibly wider

### Why zero-pad to a power of 2? Does zero-padding change the spectrum?

Above we used `next_power_of_two(N)`. That's because the Cooley–Tukey FFT from L39 **only supports power-of-2 lengths** (such as 256, 512, 1024). If the input length isn't a power of 2, it has to be zero-padded.

**Does zero-padding change the spectrum?**
It doesn't change the frequency content of the original signal: what you pad with is **no new information** (all zeros). Mathematically, padding to length M just adds terms that are zero:

$$\text{FFT}_M([x_0, x_1, \ldots, x_{N-1}, 0, 0, \ldots, 0])[k] = \sum_{n=0}^{N-1} x_n e^{-2\pi i kn/M} + \underbrace{0 + \cdots + 0}_{\text{the padded zeros}}$$

What zero-padding does is **make the frequency grid finer** — it samples the same underlying spectrum at more points. If the original data has length 1000 and sr=8000, then Δf = 8 Hz/bin. Pad to 1024 and Δf = 7.8 Hz/bin (a little finer); pad to 2048 and Δf = 3.9 Hz/bin (finer still).

**What does a "finer grid" mean concretely? — back to "each bin covers a range"**

Section 1 showed that each bin isn't an exact point but covers the range `center ± Δf/2`. Zero-padding makes **those ranges narrower**.

For example: with sr=8000 and N=1000, Δf=8 Hz/bin, so a bin covers ±4 Hz, 8 Hz wide; pad to N=2048 and Δf=3.9 Hz/bin — the same stretch of frequencies is now cut into two narrower cells, each covering only ±1.95 Hz. But careful: narrower cells only let you read off a peak's position more finely. Two frequencies that were crammed into one 8 Hz cell (say 440 Hz and 442 Hz) still blur into a single peak after padding, because the width of each peak is set by the original 1000 samples (0.125 s), not by the grid. To really tell them apart you need a longer recording — **the cells get finer, but the true resolution doesn't change** (the point L39 made: physical resolution = sr / the original number of samples).

**But watch out for spectral leakage**:
If the signal length N_demo isn't a whole number of periods of the frequency f, the cut at the ends creates a discontinuity, which produces "side lobes" in the spectrum — some energy leaks out into the bins around the target frequency. This can shift the peak and reduce accuracy.

**Remedies**:
1. Option A (windowing): smooth the ends with a Hann/Hamming window (the topic of the next lesson, L41)
2. Option B (adjusting the duration): choose a duration that holds a whole number of periods, e.g. for 440 Hz take duration = k/440 seconds (k a whole number) — and skip the zero-padding, which would bring the discontinuity back

In [ ]:
sr = 8000
duration = 0.5          # ← change to 0.01 to watch the frequency resolution get worse
x_mix = sine(440, duration, sr) + sine(880, duration, sr)
N = len(x_mix)

# guard: bail out early if frequency_bins isn't implemented, so nothing crashes
try:
    _fb_check = frequency_bins(N, sr)
except (NotImplementedError, TypeError):
    _fb_check = None
if _fb_check is None:
    print('⬜ Implement frequency_bins first, then run this cell')
else:
    # aurora_fft needs a power of 2, so zero-pad to next_power_of_two
    N_fft = next_power_of_two(N)
    x_padded = np.zeros(N_fft)
    x_padded[:N] = x_mix
    X_mix = aurora_fft(x_padded)  # the L39-style FFT
    assert np.allclose(X_mix, np.fft.fft(x_padded)), "aurora_fft consistency check"
    freqs = frequency_bins(N_fft, sr)
    amp = np.abs(X_mix[:N_fft // 2 + 1])

    plt.figure(figsize=(9, 3))
    plt.plot(freqs, amp)
    plt.axvline(440, color='r', linestyle='--', label='440 Hz')
    plt.axvline(880, color='g', linestyle='--', label='880 Hz')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude')
    plt.title(f'Magnitude spectrum of the mix  (N={N_fft}, Δf={sr/N_fft:.2f} Hz/bin)')
    plt.legend()
    plt.xlim(0, 1200)
    plt.tight_layout()
    plt.show()

    # find the frequencies of the peak bins
    # note: if the signal isn't a whole number of periods long, side lobes can shift the peak (spectral leakage);
    # a duration of k/440 s (k whole) gives whole periods of 440 Hz, though zero-padding to a power of 2 breaks that again.
    top2_idx = np.argsort(amp)[-2:][::-1]
    print('the two strongest frequencies detected:', sorted(freqs[top2_idx].astype(int)), 'Hz')

In [ ]:
# Spectral leakage compared: a signal with / without a whole number of periods
sr = 8000
freq = 440

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

# Case 1: duration isn't a whole number of periods → obvious leakage
duration_leak = 0.5
n_samples_leak = int(sr * duration_leak)
t = np.arange(n_samples_leak) / sr
x_leak = np.sin(2 * np.pi * freq * t)
N_fft = next_power_of_two(n_samples_leak)
x_padded = np.zeros(N_fft)
x_padded[:n_samples_leak] = x_leak
X_leak = np.fft.fft(x_padded)
freqs = np.arange(N_fft // 2 + 1) * sr / N_fft
amp_leak = np.abs(X_leak[:N_fft // 2 + 1])

axes[0, 0].plot(freqs, amp_leak)
axes[0, 0].set_xlim(300, 600)
axes[0, 0].set_title(f'duration={duration_leak:.3f}s (not whole periods)\nclear leakage, wider peak')
axes[0, 0].set_ylabel('Magnitude')

# Case 2: adjusted to a whole number of periods, and no zero-padding → leakage gone
# for 440 Hz, one period is 1/440 ≈ 0.00227 s
# take N_periods=110, total duration = 110/440 = 0.25 s (an even number of periods)
n_periods = 110
duration_clean = n_periods / freq
n_samples_clean = int(sr * duration_clean)
t_clean = np.arange(n_samples_clean) / sr
x_clean = np.sin(2 * np.pi * freq * t_clean)
N_fft_clean = n_samples_clean  # no zero-padding: padding would break the whole periods again
X_clean = np.fft.fft(x_clean)
freqs_clean = np.arange(N_fft_clean // 2 + 1) * sr / N_fft_clean
amp_clean = np.abs(X_clean[:N_fft_clean // 2 + 1])

axes[0, 1].plot(freqs_clean, amp_clean)
axes[0, 1].set_xlim(300, 600)
axes[0, 1].set_title(f'duration={duration_clean:.4f}s ({n_periods} periods)\nno zero-padding: leakage gone, sharp peak')
axes[0, 1].set_ylabel('Magnitude')

# zoomed comparison: side lobes in the leaky case
axes[1, 0].semilogy(freqs, np.maximum(amp_leak, 1e-6))
axes[1, 0].set_xlim(200, 700)
axes[1, 0].axvline(440, color='r', linestyle='--', alpha=0.5, label='ideal frequency')
axes[1, 0].set_title('not whole periods (log scale)')
axes[1, 0].set_xlabel('Frequency (Hz)')
axes[1, 0].set_ylabel('Magnitude')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend()

# the whole-periods case for comparison
axes[1, 1].semilogy(freqs_clean, np.maximum(amp_clean, 1e-6))
axes[1, 1].set_xlim(200, 700)
axes[1, 1].axvline(440, color='r', linestyle='--', alpha=0.5, label='ideal frequency')
axes[1, 1].set_title('whole periods, no padding (log scale)')
axes[1, 1].set_xlabel('Frequency (Hz)')
axes[1, 1].set_ylabel('Magnitude')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("Spectral leakage compared, in summary:")
print("="*60)
print(f"Not whole periods (duration={duration_leak:.3f}s):")
print(f"  - clear side lobes around the peak")
print(f"  - the energy is spread over several bins")
print(f"\nWhole periods (duration={duration_clean:.4f}s, {n_periods} periods):")
print(f"  - nearly all the energy sits in one bin")
print(f"  - the side lobes drop to the noise floor")
print("\n💡 Practical advice:")
print("   1. If you know the signal's frequency, choose a duration with whole periods (and don't zero-pad)")
print("   2. Or learn windowing in the next lesson (L41), which suppresses the side lobes automatically")

## Wrap-up

`frequency_bins(N, sr)` translates FFT bin indices into Hz, giving a one-sided frequency axis of shape `(N//2+1,)` whose last point lands exactly on the Nyquist frequency `sr/2`.
This axis is what `aurora.audio.stft.magnitude_spectrogram()` uses to label the frequencies of every frame, so what you've written here is really the coordinate layer of the STFT.
The magnitude, power and dB spectra each suit different downstream processing; in audio feature extraction, the dB spectrum is still the most common.
Next lesson (L41) layers a Hann/Hamming window onto this spectrum pipeline and runs it end to end: "signal → window → FFT → magnitude spectrum".

---
⬇️ **Completion check**: you've read the wrap-up; please finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: spectrum analysis by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: `sr=8000, N=256`
- Δf = sr/N = ? (Hz/bin)
- Which bin does 440 Hz land in? (k = round(440/Δf))
- What is that bin's exact frequency in Hz?

**Question 2**: which 5 frequency values should `frequency_bins(8, 8000)` return?
(Shape: (N//2+1,) = (5,), from 0 to Nyquist=4000 Hz)

**Question 3**: to tell apart two notes 10 Hz apart, what is the minimum number of FFT points N?
(We need Δf = sr/N ≤ 10 Hz, with sr=16000)
**Clarification**: "tell apart" means the minimum distance between the two frequency peaks is ≥ Δf, i.e. at the very least the two peaks can't sit in the same bin.

**Question 4**: how are the magnitude spectrum and the power spectrum related?
(Hint: power spectrum = magnitude spectrum²)

**Question 5**: why does the frequency axis keep only `k=0..N//2` (a one-sided spectrum) and not `k=0..N-1`?

**Question 6 (advanced)**: in Question 3, the theoretical minimum is N=1600 (giving Δf=10 Hz). But the code actually uses N=2048. Why?
**Background**: the FFT (Fast Fourier Transform) needs N to be a power of 2; the DFT (Discrete Fourier Transform) has no such restriction.
**Answer**: L39 implements the Cooley–Tukey FFT (power-of-2 lengths only), so N has to be rounded up to a power of 2 → $N = 2^{\lceil \log_2(1600) \rceil} = 2048$.
If you use a standard DFT instead (numpy.fft.fft picks an algorithm automatically), N=1600 works too.

Once you've done the derivations, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: sr=8000, N=256
sr, N = 8000, 256
delta_f = sr / N
assert delta_f == 31.25
k_440 = round(440 / delta_f)
f_k440 = k_440 * delta_f
assert k_440 == 14
assert f_k440 == 437.5
print(f"Q1 ✅  Δf={delta_f} Hz/bin, 440Hz → bin {k_440}, exact frequency={f_k440} Hz")

# Q2: frequency_bins(8, 8000) = [0, 1000, 2000, 3000, 4000]
expected_bins = np.array([0.0, 1000.0, 2000.0, 3000.0, 4000.0])
try:
    result = frequency_bins(8, 8000)
    assert result.shape == (5,), f"shape should be (5,), got {result.shape}"
    assert np.allclose(result, expected_bins, atol=1e-10)
    print(f"Q2 ✅  frequency_bins(8, 8000)={result}")
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement frequency_bins() first, then run the answer-check cell")

# Q3: minimum N for Δf≤10 Hz
sr2 = 16000
min_N = int(np.ceil(sr2 / 10))
# round up to a power of 2 (a Cooley-Tukey requirement)
N_pow2 = 2 ** int(np.ceil(np.log2(min_N)))
assert min_N == 1600
print(f"Q3 ✅  theoretical minimum N={min_N} (Δf={sr2/min_N} Hz),")
print(f'     "tell apart" = the two frequency peaks land in different bins (bin distance ≥ Δf)')
print(f"     rounded up to a power of 2, N={N_pow2}")

# Q4: magnitude spectrum vs power spectrum
X = np.fft.rfft(np.array([1.0, 0.0, -1.0, 0.0]))
mag = np.abs(X)
power = mag ** 2
assert np.allclose(power, mag**2, atol=1e-15)
print(f"Q4 ✅  magnitude spectrum={np.round(mag,4)}, power spectrum=magnitude²={np.round(power,4)}")

# Q5: one-sided spectrum — conjugate symmetry of real signals
x_real = np.random.randn(8)
X_full = np.fft.fft(x_real)
# check X[N-k] = conj(X[k])
for k_val in range(1, 4):
    assert np.isclose(X_full[8-k_val], np.conj(X_full[k_val]), atol=1e-10)
print(f"Q5 ✅  a real signal satisfies X[N-k]=X[k]* (conjugate symmetry), so the second half adds nothing new; keep only k=0..N/2")

# Q6: FFT vs DFT
print(f"\nQ6 ✅  How the FFT and the DFT differ:")
print(f"     - FFT (Fast Fourier Transform, Cooley-Tukey): needs N = 2^k, fast, O(N log N)")
print(f"     - DFT (Discrete Fourier Transform): any N works, slower, O(N²)")
print(f"     - aurora.audio.transforms.fft, the L39-style FFT, is Cooley-Tukey, so it must be zero-padded to a power of 2")
print(f"     - with numpy.fft.fft (an optimized mixed algorithm) instead, N=1600 can be computed directly, no padding needed")

print("\n🎉 Spectrum-analysis whiteboard challenge passed! frequency_bins(N,sr)=k*sr/N, k=0..N//2.")

In [ ]:
# ✏️ Self-review for this lesson
l40_review = {
    "frequency_resolution":    None,  # remember Δf=sr/N (Hz/bin)? True/False
    "frequency_bins_impl":     None,  # frequency_bins implemented and passing the assertions? True/False
    "magnitude_vs_power":      None,  # understand magnitude spectrum=|X[k]|, power spectrum=|X[k]|²? True/False
    "one_sided_spectrum":      None,  # understand that a real signal only needs a one-sided spectrum (conjugate symmetry)? True/False
    "whiteboard_passed":       None,  # whiteboard challenge derived on paper? True/False
}

unfilled = [k for k, v in l40_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l40_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L40 fully cleared! On to L41: the windowed-FFT pipeline')

---

→ **Next lesson** · [L41 · The windowed-FFT pipeline](L41_fft_full.ipynb)

> Next lesson covers **the windowed-FFT pipeline**: signal → window → FFT → magnitude spectrum, one pipeline end to end.